# Просмотр интро (`#о_себе`)

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd
import html as _html
from IPython.display import HTML, display

In [3]:
def render_df_record(row: pd.Series) -> str:
    # Хелпер для безопасного извлечения строк (заменяет NaN на пустую строку)
    def clean_str(col_name):
        val = row[col_name]
        return "" if pd.isna(val) else str(val).strip()

    first_name = clean_str("first_name")
    last_name = clean_str("last_name")
    username_val = clean_str("username")
    stack_val = clean_str("stack")
    projects_val = clean_str("projects")
    text_val = clean_str("text")
    date_val = clean_str("message_date")

    # Собираем имя и юзернейм
    name = " ".join(filter(None, [first_name, last_name]))
    username = f"@{username_val}" if username_val else "(без username)"

    # Формируем мета-данные
    meta_bits = []
    if stack_val:
        meta_bits.append(f"stack: {_html.escape(stack_val)}")
    if projects_val:
        meta_bits.append(f"projects: {_html.escape(projects_val)}")
    meta = " &nbsp;|&nbsp; ".join(meta_bits)

    text = _html.escape(text_val)

    return f"""
<div style="border:1px solid #ccc; border-radius:8px; padding:10px 14px; margin:10px 0; text-align: left;">
  <div>
    <b>{_html.escape(username)}</b> {_html.escape(name)}
    <span style="color:#aaa; font-size:0.85em; float:right;">{_html.escape(date_val)}</span>
  </div>
  <div style="color:#888; font-size:0.9em; margin-top:2px;">{meta}</div>
  <pre style="white-space:pre-wrap; font-family:inherit; margin-top:8px; background: none; border: none; padding: 0;">{text}</pre>
</div>
"""


def display_df_group(level: str, sub_df: pd.DataFrame):
    """Принимает уровень и срез датафрейма, красиво рендерит его в Jupyter Notebook."""
    if sub_df.empty:
        display(HTML(f"<h3>{_html.escape(str(level))} — 0 записей</h3>"))
        return

    header = f"<h3>{_html.escape(str(level))} — {len(sub_df)} записей</h3>"

    # Итерируемся по строкам куска датафрейма
    records_html = "".join(render_df_record(row) for _, row in sub_df.iterrows())

    # Выводим готовый HTML в интерфейс ноутбука
    display(HTML(header + records_html))


def display_filtered_group(df: pd.DataFrame, title: str, col: str, value: str):
    """Автоматически фильтрует датафрейм по вхождению строки (регистронезависимо)

    и красиво выводит результат.
    """
    # Фильтруем: приводим к строке, ищем подстроку, игнорируем регистр и NaN
    sub_df = df[df[col].astype(str).str.contains(value, case=False, na=False)]

    # Вызываем наш старый красивый рендер
    display_df_group(level=title, sub_df=sub_df)

In [4]:
NOTEBOOK_DIR = Path.cwd()
DB_PATH = NOTEBOOK_DIR.parent / "hihikvadrat.db"
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql("SELECT * FROM intros", conn)

In [6]:
df.head()

,message_id,user_id,username,first_name,last_name,message_date,text,fetched_at,level,stack,projects
0,604,1416437137,virtuozzz10,Владимир,Москвин,2025-02-02T16:37:08+00:00,👋 Всем привет! Здесь вы найдёте классное анали...,2026-07-11T10:28:00.781811+00:00,service_message,NaN,NaN
1,621,1048419640,NaN,айыылаах,NaN,2025-02-02T17:54:33+00:00,#о_себе\nВсем привет!\nМеня зовут Матвей\nХочу...,2026-07-11T10:28:00.781599+00:00,middle,"System analyst, BI analyst, Data analyst",NaN
2,623,802510572,med_d_m,Daulet,Medeubayev,2025-02-02T18:07:29+00:00,"#о_себе\nПривет! Зовут Даулет, мне 23, живу в ...",2026-07-11T10:28:00.781351+00:00,middle,Product analyst,Teez
3,682,1091914476,mahamatrahim,Борис,Кузнечик,2025-02-02T19:49:06+00:00,#о_себе \n\n1.бобах 2.мотив - lurking 3. зеро ...,2026-07-11T10:28:00.781118+00:00,unknown,NaN,NaN
4,685,1323671237,Alinka_Red,Alinka,NaN,2025-02-02T19:50:51+00:00,"Всем привет! Меня зовут Алина, я из НН, просто...",2026-07-11T10:28:00.780898+00:00,student,NaN,IT Lab


In [7]:
df.groupby("level").agg(cnt=("user_id","count")).sort_values("cnt")

,cnt
level,
lead,30
service_message,49
senior,61
not_analyst,71
middle,72
unknown,79
junior,102
student,150


In [9]:
display_filtered_group(df, title="middle", col="level", value="middle")